# Лабораторная работа № 2. Индексация, фильтрация и поиск аномалий

Кейс: поиск финансовых аномалий. В обязательных заданиях нет циклов `for`/`while`, Pandas и ручного перебора - только срезы, булевы маски и агрегации NumPy. После кода каждого задания приведены аналитические выводы.

In [1]:
import numpy as np

np.set_printoptions(suppress=True, linewidth=120)


def section(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)

## Исходные данные

In [2]:
order_id = np.arange(2001, 2013)

region = np.array([
    "Bishkek", "Osh", "Bishkek", "Karakol",
    "Osh", "Tokmok", "Bishkek", "Osh",
    "Karakol", "Tokmok", "Bishkek", "Osh"
])

quantity = np.array([2, 1, 5, 3, 2, 8, 1, 4, 2, 3, 6, 1])

unit_price = np.array([
    1500, 20000, 800, 5000, 1200, 700,
    25000, 3000, 4500, 2000, 900, 18000
])

discount = np.array([
    0.05, 0.10, 0.00, 0.15, 0.05, 0.20,
    0.30, 0.10, 0.00, 0.05, 0.50, 0.10
])

unit_cost = np.array([
    1000, 16000, 600, 4000, 900, 500,
    18000, 2200, 3500, 1500, 800, 14000
])

revenue = quantity * unit_price * (1 - discount)
profit = revenue - quantity * unit_cost
margin = profit / revenue                      # рентабельность заказа (revenue > 0 у всех)
n_orders = order_id.size

# Все массивы одной длины и в одном порядке - значит, одна маска годится для всех.
assert (region.shape == quantity.shape == unit_price.shape
        == discount.shape == unit_cost.shape == revenue.shape == profit.shape)

section("Справочно: расчётные показатели")
print("order_id:", order_id)
print("revenue :", revenue)
print("profit  :", profit)
print("margin  :", margin.round(3))
print(f"Итого выручка = {revenue.sum():,.0f}, прибыль = {profit.sum():,.0f}")


Справочно: расчётные показатели
order_id: [2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012]
revenue : [ 2850. 18000.  4000. 12750.  2280.  4480. 17500. 10800.  9000.  5700.  2700. 16200.]
profit  : [  850.  2000.  1000.   750.   480.   480.  -500.  2000.  2000.  1200. -2100.  2200.]
margin  : [ 0.298  0.111  0.25   0.059  0.211  0.107 -0.029  0.185  0.222  0.211 -0.778  0.136]
Итого выручка = 106,260, прибыль = 10,360


## Задание 1. Первичный отбор

In [3]:
section("Задание 1. Первичный отбор (отдельная маска на каждое условие)")

REVENUE_THRESHOLD = 10_000
chosen_region = "Osh"

mask_high_revenue = revenue > REVENUE_THRESHOLD
mask_qty_ge_3 = quantity >= 3
mask_region = region == chosen_region

print(f"revenue > {REVENUE_THRESHOLD:,}:", mask_high_revenue)
print("  заказы:", order_id[mask_high_revenue], "| выручка:", revenue[mask_high_revenue])
print("quantity >= 3:", mask_qty_ge_3)
print("  заказы:", order_id[mask_qty_ge_3])
print(f"region == '{chosen_region}':", mask_region)
print("  заказы:", order_id[mask_region])


Задание 1. Первичный отбор (отдельная маска на каждое условие)
revenue > 10,000: [False  True False  True False False  True  True False False False  True]
  заказы: [2002 2004 2007 2008 2012] | выручка: [18000. 12750. 17500. 10800. 16200.]
quantity >= 3: [False False  True  True False  True False  True False  True  True False]
  заказы: [2003 2004 2006 2008 2010 2011]
region == 'Osh': [False  True False False  True False False  True False False False  True]
  заказы: [2002 2005 2008 2012]


## Задание 2. Составные условия

In [4]:
section("Задание 2. Составные условия")

mask_disc_20 = discount >= 0.20
mask_rev_10k = revenue > 10_000
mask_qty_ge_4 = quantity >= 4

pair_a = mask_disc_20 & mask_rev_10k
pair_b = mask_disc_20 & mask_qty_ge_4
pair_c = mask_rev_10k & mask_qty_ge_4

print("скидка >= 20% И выручка > 10 000   :", order_id[pair_a])
print("скидка >= 20% И количество >= 4    :", order_id[pair_b])
print("выручка > 10 000 И количество >= 4 :", order_id[pair_c])

print("\nВыбранная комбинация: скидка >= 20% И количество >= 4 (аудит объёмных скидок)")
print("  заказы:", order_id[pair_b])
print("  скидка:", discount[pair_b], "| прибыль:", profit[pair_b], "| рентабельность:", margin[pair_b].round(3))
print("Обоснование: цель - проверить политику скидок на крупных партиях. Глубокая скидка\n"
      "на одну штуку - это разовая уступка, а глубокая скидка на партию из 4+ штук\n"
      "масштабируется и сильнее всего бьёт по марже. Пара 'скидка + выручка' нашла бы\n"
      "только 2007 и пропустила 2011 - самый убыточный заказ; пара 'выручка + количество'\n"
      "про скидки вообще ничего не говорит. Ограничение выбранного правила: 2007\n"
      "(скидка 30%, но 1 шт.) сюда не попадает - его ловят правила из задания 5.")


Задание 2. Составные условия
скидка >= 20% И выручка > 10 000   : [2007]
скидка >= 20% И количество >= 4    : [2006 2011]
выручка > 10 000 И количество >= 4 : [2008]

Выбранная комбинация: скидка >= 20% И количество >= 4 (аудит объёмных скидок)
  заказы: [2006 2011]
  скидка: [0.2 0.5] | прибыль: [  480. -2100.] | рентабельность: [ 0.107 -0.778]
Обоснование: цель - проверить политику скидок на крупных партиях. Глубокая скидка
на одну штуку - это разовая уступка, а глубокая скидка на партию из 4+ штук
масштабируется и сильнее всего бьёт по марже. Пара 'скидка + выручка' нашла бы
только 2007 и пропустила 2011 - самый убыточный заказ; пара 'выручка + количество'
про скидки вообще ничего не говорит. Ограничение выбранного правила: 2007
(скидка 30%, но 1 шт.) сюда не попадает - его ловят правила из задания 5.


### Выводы: Отбор и составные условия

Отдельные маски: `revenue > 10 000` → 2002, 2004, 2007, 2008, 2012; `quantity >= 3` → 2003, 2004, 2006, 2008, 2010, 2011;
`region == "Osh"` → 2002, 2005, 2008, 2012.

Три пары составных условий:

| Условие | Заказы |
|---|---|
| скидка ≥ 20% И выручка > 10 000 | 2007 |
| **скидка ≥ 20% И количество ≥ 4** | **2006, 2011** |
| выручка > 10 000 И количество ≥ 4 | 2008 |

**Выбрана пара «скидка ≥ 20% И количество ≥ 4»** (аудит скидок на крупных партиях): глубокая скидка на партию
масштабируется на много единиц и сильнее всего бьёт по марже. Эта пара нашла 2011 — самый убыточный заказ, который пара
«скидка + выручка» пропустила бы. Ограничение: заказ 2007 (скидка 30%, 1 шт.) она не ловит — его выявляют правила из раздела 5.

## Задание 3. Поиск убыточных заказов

In [5]:
section("Задание 3. Убыточные заказы")

loss_mask = profit < 0
loss_share = loss_mask.mean()                   # доля True = доля убыточных
print("loss_mask:", loss_mask)
print("Убыточные заказы:", order_id[loss_mask])
print("Выручка убыточных:", revenue[loss_mask], "| прибыль:", profit[loss_mask])
print(f"Доля убыточных: {loss_mask.sum()} из {n_orders} = {loss_share:.1%}")
print(f"Суммарный убыток: {profit[loss_mask].sum():,.0f} "
      f"({profit[loss_mask].sum() / profit[~loss_mask].sum():.1%} от прибыли прибыльных заказов)")

top_rev_idx = np.argsort(revenue)[::-1][:3]
print("\nТоп-3 по выручке:", order_id[top_rev_idx], revenue[top_rev_idx],
      "-> прибыль:", profit[top_rev_idx])
print("Вывод: заказ 2007 - второй по выручке (17 500), но убыточный: прибыль = выручка - себестоимость,\n"
      "а скидка 30% при цене 25 000 снизила цену ниже себестоимости 18 000. Высокая выручка\n"
      "говорит только об обороте; прибыль зависит от соотношения цены, скидки и себестоимости.")


Задание 3. Убыточные заказы
loss_mask: [False False False False False False  True False False False  True False]
Убыточные заказы: [2007 2011]
Выручка убыточных: [17500.  2700.] | прибыль: [ -500. -2100.]
Доля убыточных: 2 из 12 = 16.7%
Суммарный убыток: -2,600 (-20.1% от прибыли прибыльных заказов)

Топ-3 по выручке: [2002 2007 2012] [18000. 17500. 16200.] -> прибыль: [2000. -500. 2200.]
Вывод: заказ 2007 - второй по выручке (17 500), но убыточный: прибыль = выручка - себестоимость,
а скидка 30% при цене 25 000 снизила цену ниже себестоимости 18 000. Высокая выручка
говорит только об обороте; прибыль зависит от соотношения цены, скидки и себестоимости.


### Выводы: Убыточные заказы

Убыточные: **2007 (−500) и 2011 (−2 100)**, то есть **2 из 12 = 16,7%**; суммарный убыток 2 600 (20% от прибыли
остальных заказов). Заказ 2007 — второй по выручке (17 500), но убыточный. Скидка 30% снизила цену 25 000 до 17 500, что ниже
себестоимости 18 000. **Высокая выручка отражает только оборот; прибыль зависит от соотношения цены, скидки и себестоимости**,
поэтому ранжирование по выручке может ставить убыточные заказы в верх списка.

## Задание 4. Анализ скидок

In [6]:
section("Задание 4. Анализ скидок")

low_disc = discount <= 0.05
high_disc = discount >= 0.20
print("Низкая скидка (<= 5%) :", order_id[low_disc], "скидки:", discount[low_disc])
print("Высокая скидка (>= 20%):", order_id[high_disc], "скидки:", discount[high_disc])

print(f"\n{'группа':<12}{'заказов':>8}{'ср. прибыль':>13}{'ср. рентаб.':>13}{'убыточных':>11}")
print(f"{'низкая':<12}{low_disc.sum():>8}{profit[low_disc].mean():>13,.0f}"
      f"{margin[low_disc].mean():>13.1%}{(loss_mask & low_disc).sum():>11}")
print(f"{'высокая':<12}{high_disc.sum():>8}{profit[high_disc].mean():>13,.0f}"
      f"{margin[high_disc].mean():>13.1%}{(loss_mask & high_disc).sum():>11}")

big_disc_weak_profit = high_disc & (margin < 0.05)
print("\nБольшая скидка + низкая/отрицательная прибыль (рентабельность < 5%):",
      order_id[big_disc_weak_profit], "| прибыль:", profit[big_disc_weak_profit])
print("Осторожно: в группах всего 5 и 3 заказа, это не статистическая выборка. Группы различаются не только\n"
      "скидкой, но и ценой, регионом и товаром (2006 - дешёвый товар при скидке 20% остаётся прибыльным).\n"
      "Сравнение показывает связь, но не доказывает, что скидка вызвала убыток.")


Задание 4. Анализ скидок
Низкая скидка (<= 5%) : [2001 2003 2005 2009 2010] скидки: [0.05 0.   0.05 0.   0.05]
Высокая скидка (>= 20%): [2006 2007 2011] скидки: [0.2 0.3 0.5]

группа       заказов  ср. прибыль  ср. рентаб.  убыточных
низкая             5        1,106        23.8%          0
высокая            3         -707       -23.3%          2

Большая скидка + низкая/отрицательная прибыль (рентабельность < 5%): [2007 2011] | прибыль: [ -500. -2100.]
Осторожно: в группах всего 5 и 3 заказа, это не статистическая выборка. Группы различаются не только
скидкой, но и ценой, регионом и товаром (2006 - дешёвый товар при скидке 20% остаётся прибыльным).
Сравнение показывает связь, но не доказывает, что скидка вызвала убыток.


### Выводы: Скидки

| Группа | Заказов | Средняя прибыль | Средняя рентабельность | Убыточных |
|---|---:|---:|---:|---:|
| Низкая (≤ 5%) | 5 | 1 106 | 23,8% | 0 |
| Высокая (≥ 20%) | 3 | −707 | −23,3% | 2 |

Большая скидка при рентабельности < 5%: 2007 и 2011. **Это наблюдаемая связь, а не причинность**: группы малы (5 и 3 заказа)
и различаются не только скидкой, но и ценой, товаром и регионом (заказ 2006 со скидкой 20% остаётся прибыльным). Чтобы
утверждать, что скидка *вызывает* убыток, нужны данные о том, какой была бы себестоимость и маржа без скидки.

## Задание 5. Подозрительные наблюдения

In [7]:
section("Задание 5. Правила выявления подозрительных заказов")

rule1 = (discount >= 0.30) & (margin < 0.05)              # глубокая скидка + почти нулевая/отрицательная маржа
rule2 = (unit_price >= 15_000) & (quantity == 1)          # очень дорогая позиция при единичной продаже
rule3 = margin < 0.10                                     # низкая рентабельность

print("Правило 1 (скидка >= 30% И рентабельность < 5%):", order_id[rule1])
print("Правило 2 (цена >= 15 000 И количество == 1)   :", order_id[rule2])
print("Правило 3 (рентабельность < 10%)                :", order_id[rule3])

hits = rule1.astype(int) + rule2.astype(int) + rule3.astype(int)
print("\nЧисло сработавших правил по заказам:", hits)
need_check = hits >= 2
print("Срабатывают >= 2 правил:", order_id[need_check])
only_price = rule2 & ~rule1 & ~rule3
print("Срабатывает только правило 2:", order_id[only_price],
      "| скидки:", discount[only_price], "| рентабельность:", margin[only_price].round(3))
print("Интерпретация:\n"
      "  - 2011 (скидка 50%, убыток -2 100) и 2007 (скидка 30%, убыток -500): требуют проверки -\n"
      "    нужно подтвердить, кто и почему согласовал скидку, не ошибка ли это в цене/скидке.\n"
      "  - 2002 и 2012: цена 18-20 тыс. за единицу при количестве 1, но скидка 10% и рентабельность\n"
      "    11-14% - обычная продажа дорогого товара, вероятнее всего объяснимо без проверки.\n"
      "  - 2004: рентабельность 5.9% при скидке 15% - наблюдение 'под присмотром', но не аномалия.")


Задание 5. Правила выявления подозрительных заказов
Правило 1 (скидка >= 30% И рентабельность < 5%): [2007 2011]
Правило 2 (цена >= 15 000 И количество == 1)   : [2002 2007 2012]
Правило 3 (рентабельность < 10%)                : [2004 2007 2011]

Число сработавших правил по заказам: [0 1 0 1 0 0 3 0 0 0 2 1]
Срабатывают >= 2 правил: [2007 2011]
Срабатывает только правило 2: [2002 2012] | скидки: [0.1 0.1] | рентабельность: [0.111 0.136]
Интерпретация:
  - 2011 (скидка 50%, убыток -2 100) и 2007 (скидка 30%, убыток -500): требуют проверки -
    нужно подтвердить, кто и почему согласовал скидку, не ошибка ли это в цене/скидке.
  - 2002 и 2012: цена 18-20 тыс. за единицу при количестве 1, но скидка 10% и рентабельность
    11-14% - обычная продажа дорогого товара, вероятнее всего объяснимо без проверки.
  - 2004: рентабельность 5.9% при скидке 15% - наблюдение 'под присмотром', но не аномалия.


### Выводы: Подозрительные заказы

| Правило | Срабатывает на |
|---|---|
| 1: скидка ≥ 30% И рентабельность < 5% | 2007, 2011 |
| 2: цена ≥ 15 000 И количество = 1 | 2002, 2007, 2012 |
| 3: рентабельность < 10% | 2004, 2007, 2011 |

- **Требуют проверки:** 2011 (скидка 50%, убыток −2 100) и 2007 (скидка 30%, убыток −500). Они попадают под ≥ 2 правил.
  Нужно выяснить, кто согласовал скидку, нет ли ошибки в цене/скидке/себестоимости, не акция ли это.
- **Объяснимы бизнес-ситуацией:** 2002 и 2012. Цена 18–20 тыс. при 1 шт. — продажа дорогой позиции; скидка 10%, рентабельность
  11–14%. Аномалией это не является.
- **Под наблюдением:** 2004 (рентабельность 5,9% при скидке 15%) — низкая маржа, но не убыток.

## Задание 6. Анализ сегментов

In [8]:
section("Задание 6. Сегменты: Bishkek и Osh")

seg_bishkek = region == "Bishkek"
seg_osh = region == "Osh"

print(f"{'регион':<10}{'заказов':>8}{'выручка':>12}{'прибыль':>10}{'убыточных':>11}")
print(f"{'Bishkek':<10}{seg_bishkek.sum():>8}{revenue[seg_bishkek].sum():>12,.0f}"
      f"{profit[seg_bishkek].sum():>10,.0f}{(loss_mask & seg_bishkek).sum():>11}")
print(f"{'Osh':<10}{seg_osh.sum():>8}{revenue[seg_osh].sum():>12,.0f}"
      f"{profit[seg_osh].sum():>10,.0f}{(loss_mask & seg_osh).sum():>11}")
print("\nBishkek: заказы", order_id[seg_bishkek], "| Osh: заказы", order_id[seg_osh])
print("Убыточные в Bishkek:", order_id[loss_mask & seg_bishkek],
      "| в Osh:", order_id[loss_mask & seg_osh])
print("Проверка согласованности: сумма по двум маскам == сумма по объединённой маске ->",
      bool(np.isclose(revenue[seg_bishkek].sum() + revenue[seg_osh].sum(),
                      revenue[seg_bishkek | seg_osh].sum())))
print("Вывод: Osh приносит в 1,7 раза больше выручки (47 280 против 27 050) и 6 680 прибыли, без убытков.\n"
      "Bishkek при том же числе заказов (4) в сумме убыточен (-750): оба убыточных заказа (2007, 2011)\n"
      "относятся к нему. Значит, проблема не в объёме продаж региона, а в ценообразовании\n"
      "и скидках по конкретным заказам. Выборка мала (по 4 заказа), поэтому это сигнал, а не доказательство.")


Задание 6. Сегменты: Bishkek и Osh
регион     заказов     выручка   прибыль  убыточных
Bishkek          4      27,050      -750          2
Osh              4      47,280     6,680          0

Bishkek: заказы [2001 2003 2007 2011] | Osh: заказы [2002 2005 2008 2012]
Убыточные в Bishkek: [2007 2011] | в Osh: []
Проверка согласованности: сумма по двум маскам == сумма по объединённой маске -> True
Вывод: Osh приносит в 1,7 раза больше выручки (47 280 против 27 050) и 6 680 прибыли, без убытков.
Bishkek при том же числе заказов (4) в сумме убыточен (-750): оба убыточных заказа (2007, 2011)
относятся к нему. Значит, проблема не в объёме продаж региона, а в ценообразовании
и скидках по конкретным заказам. Выборка мала (по 4 заказа), поэтому это сигнал, а не доказательство.


### Выводы: Сегменты

| Регион | Заказов | Выручка | Прибыль | Убыточных |
|---|---:|---:|---:|---:|
| Bishkek | 4 | 27 050 | −750 | 2 |
| Osh | 4 | 47 280 | 6 680 | 0 |

Масштаб согласован с масками: сумма по двум региональным маскам равна сумме по объединённой маске. Osh приносит в 1,7 раза больше
выручки и без убытков; оба убыточных заказа принадлежат Bishkek, который в сумме убыточен. Проблема не в объёме продаж региона,
а в ценообразовании отдельных заказов. Выборка мала (по 4 заказа), поэтому это сигнал, а не доказательство.

## Задание 7. Влияние аномалий

In [9]:
section("Задание 7. Влияние исключения подозрительных заказов")

mean_all = revenue.mean()
median_all = np.median(revenue)
print(f"Все заказы ({n_orders}): средняя выручка = {mean_all:,.1f}, медиана = {median_all:,.1f}")

print(f"\n{'правило исключения':<34}{'исключено':>10}{'ср. выручка':>13}{'изменение':>11}{'медиана':>10}")
# Для каждого правила явно строим маску "оставить" и считаем статистику без перебора.
r1_keep, r3_keep, r2_keep, big_keep = ~rule1, ~rule3, ~rule2, ~(revenue > 15_000)
print(f"{'Правило 1 (скидка + маржа)':<34}{rule1.sum():>10}{revenue[r1_keep].mean():>13,.1f}"
      f"{revenue[r1_keep].mean() / mean_all - 1:>11.1%}{np.median(revenue[r1_keep]):>10,.0f}")
print(f"{'Правило 3 (маржа < 10%)':<34}{rule3.sum():>10}{revenue[r3_keep].mean():>13,.1f}"
      f"{revenue[r3_keep].mean() / mean_all - 1:>11.1%}{np.median(revenue[r3_keep]):>10,.0f}")
print(f"{'Правило 2 (дорогая 1 шт.)':<34}{rule2.sum():>10}{revenue[r2_keep].mean():>13,.1f}"
      f"{revenue[r2_keep].mean() / mean_all - 1:>11.1%}{np.median(revenue[r2_keep]):>10,.0f}")
print(f"{'Просто выручка > 15 000':<34}{(revenue > 15_000).sum():>10}{revenue[big_keep].mean():>13,.1f}"
      f"{revenue[big_keep].mean() / mean_all - 1:>11.1%}{np.median(revenue[big_keep]):>10,.0f}")

print("\nИтоговая прибыль до/после исключения правила 1:",
      f"{profit.sum():,.0f} -> {profit[r1_keep].sum():,.0f}")
print("Вывод: результат очень чувствителен к правилу. Правило 1 почти не меняет среднюю выручку\n"
      "(-2.8%: исключены 2 заказа), но заметно меняет итоговую прибыль (+25%) - потому что\n"
      "убирает убыточные заказы. Правило 2 снижает среднюю выручку на ~31%, так как удаляет самые\n"
      "крупные ЗАКОНОМЕРНЫЕ продажи. Удалять заказы только за большое значение нельзя: 2002 и 2012\n"
      "прибыльны, и их исключение исказило бы картину. Фильтровать нужно по признаку проблемы\n"
      "(аномальная маржа/скидка), а не по размеру.")


Задание 7. Влияние исключения подозрительных заказов
Все заказы (12): средняя выручка = 8,855.0, медиана = 7,350.0

правило исключения                 исключено  ср. выручка  изменение   медиана
Правило 1 (скидка + маржа)                 2      8,606.0      -2.8%     7,350
Правило 3 (маржа < 10%)                    3      8,145.6      -8.0%     5,700
Правило 2 (дорогая 1 шт.)                  3      6,062.2     -31.5%     4,480
Просто выручка > 15 000                    3      6,062.2     -31.5%     4,480

Итоговая прибыль до/после исключения правила 1: 10,360 -> 12,960
Вывод: результат очень чувствителен к правилу. Правило 1 почти не меняет среднюю выручку
(-2.8%: исключены 2 заказа), но заметно меняет итоговую прибыль (+25%) - потому что
убирает убыточные заказы. Правило 2 снижает среднюю выручку на ~31%, так как удаляет самые
крупные ЗАКОНОМЕРНЫЕ продажи. Удалять заказы только за большое значение нельзя: 2002 и 2012
прибыльны, и их исключение исказило бы картину. Фильтровать нужно 

### Выводы: Влияние аномалий

Все заказы: средняя выручка 8 855, медиана 7 350.

| Правило исключения | Исключено | Средняя выручка | Изменение | Медиана |
|---|---:|---:|---:|---:|
| 1 (скидка + маржа) | 2 | 8 606 | −2,8% | 7 350 |
| 3 (маржа < 10%) | 3 | 8 146 | −8,0% | 5 700 |
| 2 (дорогая единичная продажа) | 3 | 6 062 | −31,5% | 4 480 |
| «выручка > 15 000» (по размеру) | 3 | 6 062 | −31,5% | 4 480 |

Результат **сильно зависит от правила**: от −2,8% до −31,5%. Правило 1 почти не меняет среднюю выручку, но меняет прибыль
(10 360 → 12 960, +25%), так как убирает убыточные заказы. Правило 2 (как и отсечение по размеру выручки) удаляет самые крупные
*нормальные* продажи и снижает среднее на треть, хотя заказы 2002 и 2012 прибыльны. Поэтому наблюдения нельзя удалять только за то,
что они большие: критерий должен описывать проблему (аномальная маржа, скидка), а не размер. Любое исключение следует фиксировать
вместе с причиной и показывать оба результата — до и после.

## Задание 8. Рекомендации

In [10]:
section("Задание 8. Краткая сводка")
print("Аномалии        :", order_id[rule1], "(скидка >= 30% при маржинальности < 5%)")
print("Под наблюдением :", order_id[rule3 & ~rule1], "(низкая маржа)")
print("Объяснимо       :", order_id[only_price], "(дорогие единичные продажи)")
print(f"Убыточных: {loss_share:.1%}; суммарный убыток {profit[loss_mask].sum():,.0f}")


Задание 8. Краткая сводка
Аномалии        : [2007 2011] (скидка >= 30% при маржинальности < 5%)
Под наблюдением : [2004] (низкая маржа)
Объяснимо       : [2002 2012] (дорогие единичные продажи)
Убыточных: 16.7%; суммарный убыток -2,600


### Выводы: Рекомендации

**Обнаруженные аномалии:** 2011 (скидка 50%, рентабельность −78%), 2007 (скидка 30%, цена ниже себестоимости); под
наблюдением 2004 (низкая маржа).

**Что влияет на финансовые выводы:** эти два заказа дают весь убыток (−2 600), а в Bishkek вся региональная прибыль
становится отрицательной (−750). Исключение или сохранение этих заказов меняет итоговую прибыль на 25%.

**Какие проверки необходимы:**
1. Убедиться, что цена, скидка и себестоимость в 2007 и 2011 введены без ошибок (в том числе единицы измерения и валюта).
2. Найти документ о согласовании скидок ≥ 30%: кто и по какой политике их утвердил.
3. Проверить, не относятся ли заказы к акции, распродаже, пробной партии или сделке с ключевым клиентом.
4. Сверить себестоимость 2007 (18 000) с учётом: она могла быть пересчитана или указана на другую дату.

**Какие данные нужны для окончательного решения:** прайс-лист и политика скидок; история цен и себестоимости; клиент и канал
продаж каждого заказа; больший объём данных (12 заказов недостаточно для статистических выводов, особенно по регионам);
данные о возвратах и затратах, выходящих за рамки валовой прибыли.

## Дополнительное задание. Два правила и их сравнение

In [11]:
section("Дополнительное задание. Сравнение двух правил")

rule_A = (discount >= 0.30) & (profit < 0)                    # узкое: бизнес-правило
rule_B = (margin < 0.10) | (unit_price >= 15_000) | (discount >= 0.20)   # широкое

print("Правило A (скидка >= 30% И профит < 0)                  :", order_id[rule_A], f"-> {rule_A.sum()} заказа")
print("Правило B (маржа < 10% ИЛИ цена >= 15 000 ИЛИ скидка >= 20%):", order_id[rule_B], f"-> {rule_B.sum()} заказов")

both = rule_A & rule_B
only_B = rule_B & ~rule_A
print("\nОбщие срабатывания (A и B):", order_id[both])
print("Только B (потенциальные ложные срабатывания):", order_id[only_B])
print(f"Доля заказов под проверкой: A = {rule_A.mean():.0%}, B = {rule_B.mean():.0%}")
print(f"Все убыточные заказы {order_id[loss_mask]} покрыты A: {bool(np.all(rule_A[loss_mask]))}, "
      f"покрыты B: {bool(np.all(rule_B[loss_mask]))}")
fp_B = only_B & (profit > 0) & (margin >= 0.10)
print("Из 'только B' с нормальной маржой >= 10% и прибылью (скорее всего, ложные):", order_id[fp_B])

print("\nВывод: правило A - точное и дешёвое для первичной проверки: оно проверяет 2 заказа и\n"
      "ловит оба убыточных, но по построению не видит заказы без убытка (2004 с маржой 5.9%).\n"
      "Правило B - широкое: оно вернёт половину всех заказов (6 из 12), включая нормальные дорогие\n"
      "продажи (2002, 2012) и скидку 20% на дешёвый товар (2006), поэтому даст слишком много\n"
      "ложных срабатываний и перегрузит проверяющих. B полезно как второй, более широкий\n"
      "'радар' для наблюдения, но не как первичный фильтр.")


Дополнительное задание. Сравнение двух правил
Правило A (скидка >= 30% И профит < 0)                  : [2007 2011] -> 2 заказа
Правило B (маржа < 10% ИЛИ цена >= 15 000 ИЛИ скидка >= 20%): [2002 2004 2006 2007 2011 2012] -> 6 заказов

Общие срабатывания (A и B): [2007 2011]
Только B (потенциальные ложные срабатывания): [2002 2004 2006 2012]
Доля заказов под проверкой: A = 17%, B = 50%
Все убыточные заказы [2007 2011] покрыты A: True, покрыты B: True
Из 'только B' с нормальной маржой >= 10% и прибылью (скорее всего, ложные): [2002 2006 2012]

Вывод: правило A - точное и дешёвое для первичной проверки: оно проверяет 2 заказа и
ловит оба убыточных, но по построению не видит заказы без убытка (2004 с маржой 5.9%).
Правило B - широкое: оно вернёт половину всех заказов (6 из 12), включая нормальные дорогие
продажи (2002, 2012) и скидку 20% на дешёвый товар (2006), поэтому даст слишком много
ложных срабатываний и перегрузит проверяющих. B полезно как второй, более широкий
'радар' для наблюд

### Выводы: Дополнительное задание

| Правило | Условие | Срабатывает на | Доля заказов |
|---|---|---|---:|
| A (узкое) | скидка ≥ 30% И прибыль < 0 | 2007, 2011 | 17% |
| B (широкое) | маржа < 10% ИЛИ цена ≥ 15 000 ИЛИ скидка ≥ 20% | 2002, 2004, 2006, 2007, 2011, 2012 | 50% |

Оба правила находят оба убыточных заказа. Лишь B даёт 4 дополнительных срабатывания, и 3 из них (2002, 2006, 2012) — прибыльные
заказы с нормальной рентабельностью (≥ 10%), то есть вероятные ложные срабатывания. Правило **A подходит для первичной проверки**:
точное, проверяет 2 заказа, и объём проверок управляем. Его слабость — оно не видит заказы с низкой, но положительной маржой
(2004). Правило **B даёт слишком много ложных срабатываний** (половина всех заказов) и перегрузит проверяющих; его
разумно использовать как второй, более широкий «радар» для наблюдения, но не как основной фильтр.